# Laboratorio 4 - Milton Beltrán

## 0. Preparación del corpus

Este notebook es **autocontenido**, igual que los de los laboratorios anteriores: repite aquí la
carga del corpus, el pipeline de normalización, las particiones y las representaciones vectoriales
del Laboratorio #3, en lugar de importarlos desde aquel notebook.

El enunciado exige reutilizar **exactamente** las mismas particiones y los mismos vectorizadores
ajustados en el Lab #3, para que la comparación entre Naive Bayes y regresión logística sea justa.
Como no se están importando sino **reconstruyendo**, eso solo vale si el resultado es idéntico, y lo
es porque todo el pipeline es determinista: `random_state=42` fijo, vectorizadores sin parámetros y
el mismo corpus de partida. Al final de cada celda crítica hay un `assert` que lo comprueba en vez de
darlo por hecho.

In [1]:
# --- 0.1 Librerías ---

# --- Librerías estándar de Python ---
from collections import Counter
import time

# --- Manejo de datos y gráficos ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# --- Nuevo en el Lab #4 ---
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier      # línea base de clase mayoritaria (Sección 7)

In [2]:
# --- 0.2 Carga del corpus y limpieza de filas inservibles ---
# Misma cadena de limpieza del Lab #3: 1217 -> 1142 -> 1140 -> 1134 documentos.

df = pd.read_csv("df_total.csv")

# i. Filas completamente duplicadas. Se eliminan antes de normalizar.
df_prep = df.drop_duplicates().reset_index(drop=True)
print("Documentos originales:", len(df))
print("Documentos tras quitar duplicados:", len(df_prep))

# ii. Documentos sin texto: no aportan nada al vectorizador.
vacios = df_prep["news"].str.strip() == ""
print("Documentos con texto vacío:", int(vacios.sum()), "->", df_prep.loc[vacios, "Type"].tolist())
df_prep = df_prep[~vacios].reset_index(drop=True)

# iii. Noticias con el mismo texto y etiquetas CONTRADICTORIAS (3 pares).
#      Se eliminan las dos copias de cada par (keep=False): sin criterio objetivo para elegir
#      la etiqueta correcta, dejar una sería inventarse la anotación. Si sobrevivieran, habría
#      fuga entre entrenamiento y prueba y un techo de exactitud imposible de alcanzar.
contradictorios = df_prep.duplicated("news", keep=False)
print("Filas con texto repetido y etiqueta contradictoria:", int(contradictorios.sum()))
df_prep = df_prep[~contradictorios].reset_index(drop=True)

print()
print("Documentos de trabajo (definitivo):", len(df_prep))
print("Categorías:", df_prep["Type"].nunique())
print()
print(df_prep["Type"].value_counts().to_string())

# El corpus del Lab #3 tenía 1,134 documentos: si aquí saliera otro número, todo lo que sigue
# dejaría de ser comparable con aquel laboratorio.
assert len(df_prep) == 1134, f"Se esperaban 1,134 documentos y hay {len(df_prep)}"

Documentos originales: 1217
Documentos tras quitar duplicados: 1142
Documentos con texto vacío: 2 -> ['Sostenibilidad', 'Macroeconomia']
Filas con texto repetido y etiqueta contradictoria: 6

Documentos de trabajo (definitivo): 1134
Categorías: 7

Type
Macroeconomia     319
Alianzas          244
Innovacion        152
Regulaciones      141
Otra              128
Sostenibilidad    124
Reputacion         26


In [3]:
# --- 0.3 Pipeline de normalización (heredado de los Laboratorios #1 y #2) ---
# tokenización -> minúsculas -> sin puntuación -> sin stopwords -> stemming

stop_es = set(stopwords.words("spanish"))
stemmer = SnowballStemmer("spanish")

# i. Tokenización
df_prep["tokens"] = df_prep["news"].apply(
    lambda t: word_tokenize(t, language="spanish")
)

# ii. Minúsculas
df_prep["tokens"] = df_prep["tokens"].apply(
    lambda lista: [tok.lower() for tok in lista]
)

# iii. Quitar puntuación
df_prep["tokens"] = df_prep["tokens"].apply(
    lambda lista: [tok for tok in lista if tok.isalpha()]
)

# iv. Quitar stopwords
df_prep["tokens"] = df_prep["tokens"].apply(
    lambda lista: [tok for tok in lista if tok not in stop_es]
)

# v. Stemming ("lematización"): tokens_norm guarda las palabras legibles, tokens_stem las raíces.
df_prep["tokens_norm"] = df_prep["tokens"]
df_prep["tokens_stem"] = df_prep["tokens_norm"].apply(
    lambda lista: [stemmer.stem(tok) for tok in lista]
)

# Los vectorizadores de scikit-learn reciben strings, no listas de tokens.
df_prep["texto_norm"] = df_prep["tokens_stem"].apply(lambda lista: " ".join(lista))


def contar(col_de_listas):
    """Aplana una columna de listas de tokens y devuelve (tokens, tipos)."""
    todos = [tok for lista in col_de_listas for tok in lista]
    return len(todos), len(set(todos))


tok, tip = contar(df_prep["tokens_stem"])
print(f"Corpus normalizado: documentos={len(df_prep):,}  tokens={tok:,}  tipos={tip:,}")
print("Columnas del DataFrame:", list(df_prep.columns))

# Mismas cifras que el Lab #3: el pipeline no cambió.
assert (tok, tip) == (298753, 13093), f"El pipeline cambió: tokens={tok:,}, tipos={tip:,}"

Corpus normalizado: documentos=1,134  tokens=298,753  tipos=13,093
Columnas del DataFrame: ['url', 'news', 'Type', 'tokens', 'tokens_norm', 'tokens_stem', 'texto_norm']


In [4]:
# --- 0.4 Particiones 70 / 15 / 15 estratificadas (las MISMAS del Lab #3) ---
# train_test_split solo parte en DOS, así que se llama dos veces:
#   1) corpus -> train (70 %) + temp (30 %)
#   2) temp   -> validación (15 %) + prueba (15 %)   <- test_size=0.5 sobre el 30 %, no 0.15
# Se parten los ÍNDICES del DataFrame, no las columnas sueltas, para poder recuperar el texto
# original de cualquier documento mal clasificado en el análisis de errores (Sección 6).

RANDOM_STATE = 42

y_total = df_prep["Type"]

idx_train, idx_temp = train_test_split(
    df_prep.index,
    test_size=0.30,
    stratify=y_total,
    random_state=RANDOM_STATE,
)

idx_val, idx_test = train_test_split(
    idx_temp,
    test_size=0.50,                 # la mitad del 30 % -> 15 % y 15 %
    stratify=y_total.loc[idx_temp],
    random_state=RANDOM_STATE,
)

X_train_txt, y_train = df_prep.loc[idx_train, "texto_norm"], y_total.loc[idx_train]
X_val_txt,   y_val   = df_prep.loc[idx_val,   "texto_norm"], y_total.loc[idx_val]
X_test_txt,  y_test  = df_prep.loc[idx_test,  "texto_norm"], y_total.loc[idx_test]

total = len(df_prep)
for nombre, conj in [("Entrenamiento", y_train), ("Validación", y_val), ("Prueba", y_test)]:
    print(f"{nombre:<15}{len(conj):>5} documentos  ({len(conj)/total:.1%})")
print(f"{'Total':<15}{len(y_train) + len(y_val) + len(y_test):>5} documentos")

# Conjuntos disjuntos que cubren el corpus completo.
assert len(set(idx_train) & set(idx_val)) == 0
assert len(set(idx_train) & set(idx_test)) == 0
assert len(set(idx_val) & set(idx_test)) == 0
assert len(idx_train) + len(idx_val) + len(idx_test) == total

# LA comprobación del laboratorio: 793 / 170 / 171 son exactamente los tamaños del Lab #3.
# Si esto fallara, cualquier comparación con el Naive Bayes de aquel laboratorio sería inválida,
# porque la diferencia de desempeño podría venir de las particiones y no del algoritmo.
assert (len(idx_train), len(idx_val), len(idx_test)) == (793, 170, 171)
print("\nParticiones idénticas a las del Laboratorio #3.")

Entrenamiento    793 documentos  (69.9%)
Validación       170 documentos  (15.0%)
Prueba           171 documentos  (15.1%)
Total           1134 documentos

Particiones idénticas a las del Laboratorio #3.


In [5]:
# --- 0.5 Representaciones vectoriales (los MISMOS vectorizadores del Lab #3) ---
# Ambos se ajustan SOLO con entrenamiento: fit_transform en train, transform en validación y
# prueba. En TF-IDF la restricción es doble, porque el fit aprende el vocabulario y además los
# IDF; calcularlos sobre el corpus completo metería en el peso de cada palabra información sobre
# en cuántos documentos de prueba aparece.

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train_txt)
X_val_bow   = bow_vectorizer.transform(X_val_txt)
X_test_bow  = bow_vectorizer.transform(X_test_txt)

tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_txt)
X_val_tfidf   = tfidf_vectorizer.transform(X_val_txt)
X_test_tfidf  = tfidf_vectorizer.transform(X_test_txt)

vocab_bow = bow_vectorizer.get_feature_names_out()

print("Matriz BoW de entrenamiento:   ", X_train_bow.shape)
print("Matriz TF-IDF de entrenamiento:", X_train_tfidf.shape)
print("Vocabulario aprendido en entrenamiento:", f"{len(vocab_bow):,} términos")
print("¿Mismo vocabulario en las dos representaciones?",
      list(vocab_bow) == list(tfidf_vectorizer.get_feature_names_out()))
print()

# Misma forma, contenido distinto: BoW guarda conteos enteros sin cota, TF-IDF pesos reales
# normalizados a norma L2 = 1. Esa diferencia de escala es la que explicará buena parte de los
# resultados de la Sección 2.
fila_bow, fila_tfidf = X_train_bow[0], X_train_tfidf[0]
print(f"Documento de entrenamiento 0 — términos distintos: {fila_bow.nnz}")
print(f"  BoW  : enteros, máximo = {int(fila_bow.data.max())}, suma = {int(fila_bow.sum())}")
print(f"  TFIDF: reales,  máximo = {fila_tfidf.data.max():.4f}, "
      f"norma L2 = {np.sqrt((fila_tfidf.data ** 2).sum()):.4f}")

# Mismo vocabulario que en el Lab #3: los vectorizadores son los mismos, no unos equivalentes.
assert len(vocab_bow) == 10968, f"El vocabulario cambió: {len(vocab_bow):,} términos"
print("\nVectorizadores idénticos a los del Laboratorio #3.")

Matriz BoW de entrenamiento:    (793, 10968)
Matriz TF-IDF de entrenamiento: (793, 10968)
Vocabulario aprendido en entrenamiento: 10,968 términos
¿Mismo vocabulario en las dos representaciones? True

Documento de entrenamiento 0 — términos distintos: 74
  BoW  : enteros, máximo = 3, suma = 93
  TFIDF: reales,  máximo = 0.2840, norma L2 = 1.0000

Vectorizadores idénticos a los del Laboratorio #3.


In [6]:
# --- 0.6 Naive Bayes del Laboratorio #3 (línea de comparación) ---
# El enunciado pide tener disponible el clasificador del laboratorio anterior. Se reentrena aquí
# en vez de copiar sus números, para que los tiempos de la Sección 4 se midan en la misma máquina.

nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)

nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)

for nombre, modelo, X in [("BoW", nb_bow, X_val_bow), ("TF-IDF", nb_tfidf, X_val_tfidf)]:
    pred = modelo.predict(X)
    print(f"Naive Bayes {nombre:<7} validación -> "
          f"accuracy {accuracy_score(y_val, pred):.4f} | "
          f"F1 macro {f1_score(y_val, pred, average='macro'):.4f}")

# Cifras del Lab #3 que este notebook debe reproducir: BoW 0.8000 / 0.7324, TF-IDF 0.5471 / 0.3602.
assert round(accuracy_score(y_val, nb_bow.predict(X_val_bow)), 4) == 0.8000
assert round(accuracy_score(y_val, nb_tfidf.predict(X_val_tfidf)), 4) == 0.5471
print("\nNaive Bayes reproduce exactamente los resultados del Laboratorio #3.")

Naive Bayes BoW     validación -> accuracy 0.8000 | F1 macro 0.7324
Naive Bayes TF-IDF  validación -> accuracy 0.5471 | F1 macro 0.3602

Naive Bayes reproduce exactamente los resultados del Laboratorio #3.


## 1. Construcción del clasificador de regresión logística

Se entrenan dos clasificadores de regresión logística sobre el mismo problema del Lab #3: uno con
la matriz de bolsa de palabras y otro con la matriz TF-IDF, ambas ya construidas en la Sección 0 a
partir del conjunto de entrenamiento únicamente.

A diferencia de Naive Bayes, que estima cada `P(wᵢ|c)` contando de forma aislada, la regresión
logística **aprende todos sus pesos a la vez**, optimizando directamente la separación entre las
siete categorías. Esta sección construye los modelos, mira qué forma tiene lo que aprendieron y
verifica paso a paso cómo se convierte un puntaje `z = w·x + b` en una probabilidad y luego en una
predicción de clase.

In [7]:
# --- 1.1 Entrenamiento de los dos modelos ---
# Se dejan los valores por defecto: solver='lbfgs', penalty='l2', C=1.0. El efecto de C se estudia
# en la Sección 5. random_state fijo por reproducibilidad.

lr_bow = LogisticRegression(random_state=RANDOM_STATE)
lr_bow.fit(X_train_bow, y_train)

lr_tfidf = LogisticRegression(random_state=RANDOM_STATE)
lr_tfidf.fit(X_train_tfidf, y_train)

# n_iter_ dice cuántas iteraciones necesitó el optimizador. Si llegara al límite (max_iter=100 por
# defecto) scikit-learn emitiría un ConvergenceWarning y los pesos no serían el óptimo, sino donde
# se quedó el optimizador: conviene comprobarlo en vez de suponerlo.
for nombre, modelo in [("BoW", lr_bow), ("TF-IDF", lr_tfidf)]:
    print(f"Regresión logística {nombre:<7} -> iteraciones {int(modelo.n_iter_[0]):>3} "
          f"de un máximo de {modelo.max_iter}  | convergió: {int(modelo.n_iter_[0]) < modelo.max_iter}")

print()
print("Ambos convergen dentro del límite por defecto, pero no cuesta lo mismo:")
print(f"  BoW necesita {lr_bow.n_iter_[0] / lr_tfidf.n_iter_[0]:.1f}× más iteraciones que TF-IDF.")

Regresión logística BoW     -> iteraciones  55 de un máximo de 100  | convergió: True
Regresión logística TF-IDF  -> iteraciones  33 de un máximo de 100  | convergió: True

Ambos convergen dentro del límite por defecto, pero no cuesta lo mismo:
  BoW necesita 1.7× más iteraciones que TF-IDF.


In [8]:
# --- 1.2 Qué aprendió el modelo, y qué forma tiene ---
# Con 7 categorías el modelo NO aprende un vector de pesos, sino uno por categoría.

print("coef_     ", lr_bow.coef_.shape, "-> (categorías, términos del vocabulario)")
print("intercept_", lr_bow.intercept_.shape, "-> un sesgo b por categoría")
print("classes_  ", list(lr_bow.classes_))
print()

# Naive Bayes guarda una matriz de la MISMA forma... pero lo que hay en cada celda es distinto.
print("Comparación con el Naive Bayes del Lab #3:")
print(f"  nb_bow.feature_log_prob_ {nb_bow.feature_log_prob_.shape} -> log P(w|c), estimado contando")
print(f"  lr_bow.coef_             {lr_bow.coef_.shape} -> pesos w, ajustados por optimización")
print()

# Diferencia clave: log P(w|c) es siempre negativo (es un logaritmo de probabilidad) y cada fila
# suma 1 al exponenciarla. Los pesos de la regresión logística no son probabilidades: pueden ser
# negativos, y un peso negativo es evidencia EN CONTRA de la categoría, algo que Naive Bayes no
# puede expresar.
print(f"log P(w|c) de Naive Bayes: mínimo {nb_bow.feature_log_prob_.min():.3f}, "
      f"máximo {nb_bow.feature_log_prob_.max():.3f}  (todos negativos)")
print(f"pesos de regresión logística: mínimo {lr_bow.coef_.min():.3f}, "
      f"máximo {lr_bow.coef_.max():.3f}")
print(f"  pesos positivos: {(lr_bow.coef_ > 0).sum():,} | "
      f"negativos: {(lr_bow.coef_ < 0).sum():,} | "
      f"exactamente cero: {(lr_bow.coef_ == 0).sum():,}")

coef_      (7, 10968) -> (categorías, términos del vocabulario)
intercept_ (7,) -> un sesgo b por categoría
classes_   ['Alianzas', 'Innovacion', 'Macroeconomia', 'Otra', 'Regulaciones', 'Reputacion', 'Sostenibilidad']

Comparación con el Naive Bayes del Lab #3:
  nb_bow.feature_log_prob_ (7, 10968) -> log P(w|c), estimado contando
  lr_bow.coef_             (7, 10968) -> pesos w, ajustados por optimización

log P(w|c) de Naive Bayes: mínimo -11.185, máximo -4.240  (todos negativos)
pesos de regresión logística: mínimo -0.824, máximo 1.013
  pesos positivos: 31,002 | negativos: 45,774 | exactamente cero: 0


In [9]:
# --- 1.3 De z a probabilidad y de probabilidad a clase, calculado a mano ---
# Se toma un documento de validación y se rehace el camino completo sin usar predict_proba,
# para comprobar que se entiende qué hace el modelo por dentro.

doc = 0
fila = X_val_bow[doc]

# i. Los siete puntajes z = w·x + b, uno por categoría. Son números sin escala: no son
#    probabilidades ni nada interpretable todavía.
z = np.asarray(fila @ lr_bow.coef_.T + lr_bow.intercept_).ravel()

# ii. Softmax: exponenciar y normalizar para que los siete valores sumen 1. Restar el máximo antes
#     de exponenciar no cambia el resultado y evita el desbordamiento de np.exp (la misma
#     protección usada en la implementación propia de Naive Bayes del Lab #3).
exp_z = np.exp(z - z.max())
probs = exp_z / exp_z.sum()

# iii. La predicción es simplemente la categoría con la probabilidad más alta.
prediccion = lr_bow.classes_[probs.argmax()]

tabla = pd.DataFrame({"z = w·x + b": z, "softmax(z)": probs}, index=lr_bow.classes_)
print(f"Documento {doc} de validación — categoría real: {y_val.iloc[doc]}")
print(tabla.sort_values("softmax(z)", ascending=False).to_string(
    formatters={"z = w·x + b": "{:.3f}".format, "softmax(z)": "{:.4f}".format}))
print(f"\nSuma de las siete probabilidades: {probs.sum():.4f}")
print(f"Predicción (argmax): {prediccion}  |  lr_bow.predict(): {lr_bow.predict(fila)[0]}")
print()

# ¿Coincide el cálculo manual con scikit-learn? Esta comprobación es la que demuestra que el
# modelo es MULTINOMIAL (softmax sobre las 7 clases) y no one-vs-rest.
print("softmax manual == predict_proba:",
      np.allclose(probs, lr_bow.predict_proba(fila).ravel()))

# Si el modelo fuese one-vs-rest, cada z pasaría por la sigmoide σ(z) = 1/(1+e^-z) por separado y
# las siete se normalizarían después. Da otros números: es la prueba de que NO es lo que ocurre.
sigmoides = 1 / (1 + np.exp(-z))
sigmoides_norm = sigmoides / sigmoides.sum()
print("sigmoide normalizada (one-vs-rest) == predict_proba:",
      np.allclose(sigmoides_norm, lr_bow.predict_proba(fila).ravel()))
print()
print("Probabilidad de la categoría ganadora:")
print(f"  softmax        : {probs.max():.4f}")
print(f"  one-vs-rest    : {sigmoides_norm.max():.4f}   (no es lo que usa el modelo)")

Documento 0 de validación — categoría real: Sostenibilidad
               z = w·x + b softmax(z)
Sostenibilidad       7.251     0.9953
Regulaciones         1.338     0.0027
Alianzas             0.593     0.0013
Otra                -0.419     0.0005
Macroeconomia       -1.033     0.0003
Reputacion          -3.523     0.0000
Innovacion          -4.207     0.0000

Suma de las siete probabilidades: 1.0000
Predicción (argmax): Sostenibilidad  |  lr_bow.predict(): Sostenibilidad

softmax manual == predict_proba: True
sigmoide normalizada (one-vs-rest) == predict_proba: False

Probabilidad de la categoría ganadora:
  softmax        : 0.9953
  one-vs-rest    : 0.3185   (no es lo que usa el modelo)


### 1.4 Resultados y discusión

#### ¿Qué representan `z = w·x + b` y la función sigmoide σ(z) = 1/(1+e⁻ᶻ)?

**`x`** es el documento ya vectorizado: una fila de 10,968 números, casi todos cero, donde cada
posición es un término del vocabulario aprendido en entrenamiento. **`w`** es el vector de pesos
que el modelo aprendió para *una* categoría, con un peso por término, y **`b`** es su sesgo, que
recoge lo que el modelo cree sobre esa categoría antes de leer el documento.

El producto punto `w·x` suma el peso de cada término tantas veces como aparece en el documento.
**`z` es un puntaje sin escala**: en el ejemplo de la celda 1.3 va de −4.2 a 7.3. Un `z` alto
significa que las palabras del documento apuntan hacia esa categoría; uno negativo, que apuntan en
contra. Pero un 7.3 no dice por sí solo qué tan seguro está el modelo, porque no está acotado.

La **sigmoide** resuelve eso: aplasta cualquier número real al intervalo (0, 1), de forma monótona
—si `z` sube, σ(z) sube— y con σ(0) = 0.5 como punto de indiferencia. Así el puntaje se vuelve
interpretable como probabilidad. Y como es una transformación creciente, no altera el orden: la
categoría con mayor `z` es también la de mayor probabilidad.

#### ¿Cómo se pasa del puntaje a una probabilidad y luego a una predicción?

Con dos clases bastaría una sigmoide sobre un único `z`. Aquí hay **siete categorías**, y por eso
el modelo aprende una matriz `coef_` de forma **(7, 10968)** más siete sesgos: un `z` por categoría.
La sigmoide se generaliza entonces a **softmax**, que exponencia los siete puntajes y los normaliza:

```
P(c | x) = exp(z_c) / Σ_k exp(z_k)
```

La diferencia con aplicar siete sigmoides por separado no es cosmética. Softmax obliga a que las
siete probabilidades **sumen exactamente 1**, así que las categorías compiten entre sí: subir la
probabilidad de una baja la de las demás. La predicción final es el `argmax`. En el documento del
ejemplo, `z = 7.25` para Sostenibilidad se convierte en una probabilidad de **0.9953**, mientras que
la sigmoide normalizada de ese mismo `z` habría dado **0.3185** — números distintos, y por eso la
comprobación con `np.allclose` de la celda 1.3 sirve como prueba de qué está haciendo el modelo.

#### El ajuste por tener más de dos categorías

El enunciado pide hacer "los ajustes necesarios tomando en cuenta que el dataset tiene más de dos
categorías". La forma aparentemente obvia sería pasar `multi_class="multinomial"`, pero **ese
parámetro está deprecado** desde scikit-learn 1.5 y se elimina en la 1.8; en la versión de este
entorno (1.7.2) pasarlo emite un `FutureWarning`. Con `multi_class='auto'`, siete clases y el
solver `lbfgs`, el modelo **ya usa softmax multinomial**, que es justamente lo que se verificó.

El ajuste real, entonces, no es escribir un parámetro sino entender que el modelo pasó de un vector
de pesos a siete y de la sigmoide a softmax. Lo que sí hay que comprobar es la **convergencia**:
`lbfgs` es un optimizador iterativo, y si se queda sin iteraciones devuelve donde se detuvo, no el
óptimo. Ambos modelos convergen dentro del límite por defecto, pero **BoW necesita 55 iteraciones
frente a 33 de TF-IDF**, porque sus conteos crudos no están acotados —un documento largo puede
repetir una raíz decenas de veces— mientras que TF-IDF normaliza cada documento a norma L2 = 1 y
deja una superficie mucho mejor escalada para optimizar.

#### La diferencia de fondo con Naive Bayes

Las dos matrices tienen la misma forma, **(7, 10968)**, y los dos clasificadores son lineales: ambos
calculan un puntaje que es una suma ponderada de los términos del documento y se quedan con el
mayor. La diferencia está en **de dónde salen esos números**.

En Naive Bayes cada `log P(wᵢ|c)` se estimó **contando de forma aislada**, bajo el supuesto de que
las palabras son independientes dadas la categoría; todos sus valores son negativos y cada fila,
exponenciada, suma 1. En regresión logística los 76,776 pesos se ajustan **conjuntamente**, y por
eso pueden ser positivos o negativos: un peso negativo es **evidencia en contra** de la categoría,
algo que la estimación aislada de `P(wᵢ|c)` no tiene forma de expresar. Esa asimetría es lo que se
explora en la Sección 3.